In [1]:
import pandas as pd
import io
from sqlalchemy import text
from sqlalchemy import create_engine


In [2]:
df_1 = pd.read_excel("./data/GenericAnomaly_dump_result_chunk_1.xlsx")
df_2 = pd.read_excel("./data/GenericAnomaly_dump_result_chunk_2.xlsx")
df_3 = pd.read_excel("./data/GenericAnomaly_dump_result_chunk_3.xlsx")
df_4 = pd.read_excel("./data/GenericAnomaly_dump_result_chunk_4.xlsx")
df_5 = pd.read_excel("./data/GenericAnomaly_dump_result_chunk_5.xlsx")
df_6 = pd.read_excel("./data/GenericAnomaly_dump_result_chunk_6.xlsx")
df_7 = pd.read_excel("./data/GenericAnomaly_dump_result_chunk_7.xlsx")
df_8 = pd.read_excel("./data/GenericAnomaly_dump_result_chunk_8.xlsx")

In [3]:
df_chunks = pd.concat([
    df_1,
    df_2,
    df_3,
    df_4,
    df_5,
    df_6,
    df_7,
    df_8,
], ignore_index=True)

In [4]:
print(df_chunks.shape[0] == df_1.shape[0] + df_2.shape[0] + df_3.shape[0] + df_4.shape[0] + df_5.shape[0] + df_6.shape[0] + df_7.shape[0] + df_8.shape[0])
print(df_chunks.shape)

True
(167138, 19)


In [5]:
# Connexion
db_connection_str = "mysql+pymysql://root:@localhost:3306/natixis"
engine = create_engine(db_connection_str)

def insert_into_sql(df, engine, table):
    try:
        df.to_sql(
            name=table,
            con=engine,
            if_exists="replace",
            index=False,
            chunksize=1000,
            method="multi",
        )
        print(f"Insertion réussie dans la table '{table}'.")
    except Exception as e:
        print(f"Erreur lors de l'insertion : {e}")


In [6]:
insert_into_sql(df_chunks, engine, "generic_anomalies")

Insertion réussie dans la table 'generic_anomalies'.


In [7]:
import pandas as pd
from sqlalchemy import create_engine


def load_and_fix_dump(filepath):
    with open(filepath, "r", encoding="latin1", errors="replace") as f:
        cleaned_lines = [line.strip().strip('"') for line in f]

    cleaned_content = "\n".join(cleaned_lines)

    df = pd.read_csv(io.StringIO(cleaned_content), sep="\t", low_memory=False)
    return df


df_dump = load_and_fix_dump("./data/anomaly_dump_result.csv")

sql_columns = [
    "anomaly_kuid",
    "title_txt",
    "description_txt",
    "business_object_typ",
    "source_application_iua_cod",
    "source_functional_anomaly_id",
    "control_id",
    "typology_id",
    "detection_time",
    "asof_dat",
    "frequency_typ",
    "priority_typ",
    "hotfix_flg",
    "hotfix_starting_asof_dat",
    "hotfix_expiration_asof_dat",
    "source_event_typ",
    "object_identification_fields",
    "error_fields",
    "other_fields",
    "creation_time",
    "update_time",
    "correction_mode_typ",
]
df_to_insert = df_dump[[c for c in df_dump.columns if c in sql_columns]].copy()


In [8]:
insert_into_sql(df_dump, engine, "anomalies")

Insertion réussie dans la table 'anomalies'.


In [9]:
def reset_and_load(file_path):
    xlsx = pd.ExcelFile(file_path)

    tables_mapping = [
        ("BusinessObject", "business_objects"),
        ("FunctionalControl", "functional_controls"),
        ("BusinessObjectField", "business_object_fields"),
        ("Typology", "typologies"),
        ("BusinessData", "business_data"),
        ("BusinessDataFieldLink", "business_data_field_link"),
    ]

    with engine.connect() as conn:
        print("🧹 Nettoyage de la base de données...")
        conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))

        for _, table_name in tables_mapping:
            conn.execute(text(f"TRUNCATE TABLE {table_name};"))
        
        conn.commit()
        print("📥 Début de l'importation...")

        for sheet_name, table_name in tables_mapping:
            if sheet_name in xlsx.sheet_names:
                df = pd.read_excel(xlsx, sheet_name=sheet_name)
                
                df.columns = [c.strip() for c in df.columns]
                df.columns = [c.replace('businessobject', 'business_object') for c in df.columns]

                pk_map = {
                    'business_objects': ['business_object_id'],
                    'business_object_fields': ['business_object_field_id'],
                    'functional_controls': ['functional_control_id'],
                    'typologies': ['typology_id'],
                    'business_data': ['business_data_id'],
                    'business_data_field_link': ['business_object_field_id', 'managing_application_iua_cod']
                }
                
                cols_to_check = pk_map.get(table_name)
                if cols_to_check and all(c in df.columns for c in cols_to_check):
                    initial_count = len(df)
                    df = df.drop_duplicates(subset=cols_to_check, keep='first')
                    if len(df) < initial_count:
                        print(f"   ⚠️ {initial_count - len(df)} doublons supprimés dans l'onglet {sheet_name}")

                if table_name == "business_object_fields" and 'nature_field_typ' in df.columns:
                    df['nature_field_typ'] = df['nature_field_typ'].fillna('DATA')
                
                for col in df.columns:
                    if "_flg" in col:
                        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(bool).astype(int)
                
                df = df.where(pd.notnull(df), None)

                try:
                    df.to_sql(name=table_name, con=conn, if_exists="append", index=False)
                    print(f"✅ {table_name} chargée ({len(df)} lignes).")
                except Exception as e:
                    print(f"❌ Erreur critique sur {table_name}: {e}")

        conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
        conn.commit()
        print("🚀 Base de données prête pour le Hackathon !")

reset_and_load("./data/Configuration.xlsx")

🧹 Nettoyage de la base de données...
📥 Début de l'importation...
   ⚠️ 3 doublons supprimés dans l'onglet BusinessObject
✅ business_objects chargée (56 lignes).
✅ functional_controls chargée (264 lignes).
   ⚠️ 5 doublons supprimés dans l'onglet BusinessObjectField
✅ business_object_fields chargée (1060 lignes).
✅ typologies chargée (264 lignes).
✅ business_data chargée (93 lignes).
✅ business_data_field_link chargée (201 lignes).
🚀 Base de données prête pour le Hackathon !
